# Data Adaptation Pipeline
Adapts Iowa and COMPAS datasets into a unified schema, checks duplicates at each step, and generates 6 output CSV files.
 
⚠️ **Disclaimer:** This specific notebook is primarily done using AI so that it can cover all the possible features existing in both the datasets. It was created solely to adapt the Iowa and COMPAS dataset files into a unified format, and is *not* related to the Unlearning algorithm or its implementation.

In [3]:
import pandas as pd
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────────────────
IOWA_PATH   = "Iowa_3-Year_Recidivism.csv"
COMPAS_PATH = "compas-scores-two-years.csv"
OUT         = ""  # Output Path

rng = np.random.default_rng(42)

## 1. Load & Basic Cleanup

In [10]:
iowa_raw   = pd.read_csv(IOWA_PATH)
compas_raw = pd.read_csv(COMPAS_PATH)
iowa_raw.columns   = [c.strip() for c in iowa_raw.columns]
compas_raw.columns = [c.strip() for c in compas_raw.columns]

print(f"Iowa raw shape:   {iowa_raw.shape}")
print(f"COMPAS raw shape: {compas_raw.shape}")

Iowa raw shape:   (26020, 12)
COMPAS raw shape: (7214, 53)


## 2. Iowa Cleaning

In [11]:
iowa = iowa_raw.rename(columns={
    "Race - Ethnicity"                      : "race",
    "Age At Release"                        : "age_cat",
    "Convicting Offense Classification"     : "offense_classification",
    "Convicting Offense Type"               : "offense_type",
    "Convicting Offense Subtype"            : "offense_subtype",
    "Recidivism - Return to Prison numeric" : "recidivism",
}).copy()

def iowa_race(r): #Having common races between Compas and Iowa
    if pd.isna(r): return np.nan
    parts     = [p.strip() for p in str(r).lower().split("-")]
    color     = parts[0].strip()
    ethnicity = parts[1].strip() if len(parts) > 1 else "" #Some data are made of combined words
    if ethnicity == "hispanic":                          return "Hispanic"
    if "black"   in color:                               return "Black"
    if "white"   in color:                               return "White"
    if "asian"   in color or "pacific"   in color:       return "Asian"
    if "american indian" in color or "alaska" in color:  return "Native American"
    return np.nan

iowa["race"] = iowa["race"].apply(iowa_race)

iowa["age_cat"] = iowa["age_cat"].map({
    "Under 25"    : "Less than 25",
    "25-34"       : "25 - 45",
    "35-44"       : "25 - 45",
    "45-54"       : "Greater than 45",
    "55 and Older": "Greater than 45",
})

def iowa_fm(x): #Reducing to F or M like in Compass
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    if "felony" in s or "special sentence" in s or "sexual predator" in s: return "F"
    if "misd" in s: return "M"
    return np.nan

iowa["offense_classification"] = iowa["offense_classification"].apply(iowa_fm)

BASE_COLS = ["race", "age_cat", "offense_classification",
             "offense_type", "offense_subtype", "recidivism"]
iowa = iowa[BASE_COLS].dropna(
    subset=["race", "age_cat", "offense_classification", "offense_type"]
).reset_index(drop=True)
iowa["source"] = "iowa" #Specifying source for every 

print(f"Iowa clean shape: {iowa.shape}")
print(iowa["offense_type"].value_counts())

Iowa clean shape: (25985, 7)
offense_type
Drug            7907
Property        7364
Violent         5805
Public Order    3601
Other           1308
Name: count, dtype: int64


### Duplicate Check — Iowa after cleaning

In [6]:
exact_dups = iowa.duplicated().sum()
feat_dups  = iowa.duplicated(subset=BASE_COLS).sum()
print(f"Iowa exact duplicates:          {exact_dups} ({exact_dups/len(iowa)*100:.1f}%)")
print(f"Iowa feature duplicates:        {feat_dups}  ({feat_dups/len(iowa)*100:.1f}%)")
iowa.head(3)

Iowa exact duplicates:          25143 (96.8%)
Iowa feature duplicates:        25143  (96.8%)


,race,age_cat,offense_classification,offense_type,offense_subtype,recidivism,source
0,White,Less than 25,F,Violent,Assault,1,iowa
1,White,Greater than 45,F,Public Order,OWI,1,iowa
2,White,25 - 45,F,Property,Burglary,1,iowa


## 3. COMPAS Adaptation

In [13]:
def compas_race(r): #To have common Races
    if pd.isna(r): return np.nan
    r = str(r).strip().lower()
    if "african" in r:  return "Black"
    if "caucasian" in r: return "White"
    if "hispanic" in r: return "Hispanic"
    if "asian" in r:    return "Asian"
    if "native" in r:   return "Native American"
    return "Other"

def desc_to_offense_type(desc): # Mapping into a offense type for IOWA
    if pd.isna(desc): return "Other"
    d = str(desc).lower()
    if any(w in d for w in ["drug", "cocaine", "heroin", "meth", "marijuana",
                              "cannabis", "controlled substance", "narcotic",
                              "alprazolam", "oxycodone", "hydrocodone"]):
        return "Drug"
    if any(w in d for w in ["murder", "homicide", "manslaughter", "kidnap",
                              "robbery", "carjacking", "assault", "battery",
                              "rape", "sexual", "molest", "stalking",
                              "threaten", "weapon", "firearm", "gun",
                              "armed", "violent", "aggravated"]):
        return "Violent"
    if any(w in d for w in ["theft", "larceny", "burglary", "shoplift",
                              "stolen", "steal", "arson", "vandal",
                              "criminal mischief", "fraud", "forgery",
                              "counterfeit", "identity", "credit card",
                              "uttering", "trespass", "tamper"]):
        return "Property"
    if any(w in d for w in ["dui", "dwi", "driving under", "impair",
                              "traffic", "driving", "license", "registration",
                              "alcohol", "intox", "drunk", "open container",
                              "disorderly", "resist", "obstruction", "loiter",
                              "prostitut", "pimp", "solicit", "escape",
                              "abscond", "flee", "probation", "parole",
                              "violation", "sex offender", "registry"]):
        return "Public Order"
    return "Other"

def desc_to_subtype(desc):  # Mapping into a offense subtytype for IOWA
    if pd.isna(desc): return "Other Criminal"
    d = str(desc).lower()
    if any(w in d for w in ["dui", "dwi", "driving under the influence", "impair"]): return "OWI"
    if any(w in d for w in ["traffic", "driving", "license", "registration", "speeding", "reckless driv"]): return "Traffic"
    if any(w in d for w in ["murder", "homicide", "manslaughter"]): return "Murder/Manslaughter"
    if "kidnap" in d: return "Kidnap"
    if any(w in d for w in ["robbery", "carjacking"]): return "Robbery"
    if any(w in d for w in ["assault", "battery"]): return "Assault"
    if any(w in d for w in ["weapon", "firearm", "gun", "ammo", "armed"]): return "Weapons"
    if "burglary" in d: return "Burglary"
    if any(w in d for w in ["stolen property", "receiving stolen", "possess stolen"]): return "Stolen Property"
    if any(w in d for w in ["theft", "larceny", "shoplift", "steal"]): return "Theft"
    if "arson" in d: return "Arson"
    if any(w in d for w in ["vandal", "criminal mischief"]): return "Vandalism"
    if any(w in d for w in ["fraud", "forgery", "counterfeit", "identity", "credit card", "uttering"]): return "Forgery/Fraud"
    if any(w in d for w in ["drug", "cocaine", "heroin", "meth", "marijuana", "cannabis", "controlled substance", "narcotic"]):
        if any(w in d for w in ["sell", "sale", "deliver", "traffick", "distribut", "manufactur", "supply"]): return "Trafficking"
        if any(w in d for w in ["possess", "possession", "purchase"]): return "Drug Possession"
        return "Other Drug"
    if any(w in d for w in ["sex", "sexual", "rape", "molest", "lewd", "child porn", "obscen"]):
        if any(w in d for w in ["register", "registry", "residency", "offender reg"]): return "Sex Offender Registry/Residency"
        return "Sex"
    if any(w in d for w in ["prostitut", "pimp", "solicit"]): return "Prostitution/Pimping"
    if any(w in d for w in ["escape", "abscond", "flee", "flight"]): return "Flight/Escape"
    if any(w in d for w in ["violation of probation", "vop", "revocation", "parole violation", "probation violation"]): return "Special Sentence Revocation"
    if any(w in d for w in ["alcohol", "intox", "drunk", "open container"]): return "Alcohol"
    if any(w in d for w in ["trespass", "disorderly", "resist", "obstruction", "loiter"]): return "Other Public Order"
    if any(w in d for w in ["aggravated", "violent", "threaten", "stalking"]): return "Other Violent"
    return "Other Criminal"

compas = pd.DataFrame({
    "race"                  : compas_raw["race"].apply(compas_race),
    "age_cat"               : compas_raw["age_cat"],
    "offense_classification": compas_raw["c_charge_degree"].astype(str).str.strip().str.upper(),
    "offense_type"          : compas_raw["c_charge_desc"].apply(desc_to_offense_type),
    "offense_subtype"       : compas_raw["c_charge_desc"].apply(desc_to_subtype),
    "priors_count"          : pd.to_numeric(compas_raw["priors_count"],    errors="coerce").clip(lower=0).round(),
    "juv_fel_count"         : pd.to_numeric(compas_raw["juv_fel_count"],   errors="coerce").clip(lower=0).round(),
    "juv_misd_count"        : pd.to_numeric(compas_raw["juv_misd_count"],  errors="coerce").clip(lower=0).round(),
    "juv_other_count"       : pd.to_numeric(compas_raw["juv_other_count"], errors="coerce").clip(lower=0).round(),
    "recidivism"            : pd.to_numeric(compas_raw["two_year_recid"],  errors="coerce"),
})

compas.loc[~compas["offense_classification"].isin(["F", "M"]), "offense_classification"] = np.nan
compas = compas.dropna(subset=["race", "age_cat", "offense_classification"]).reset_index(drop=True)
compas["source"] = "compas"

print(f"COMPAS adapted shape: {compas.shape}")
print(compas["offense_type"].value_counts())

COMPAS adapted shape: (7214, 11)
offense_type
Violent         1985
Other           1920
Property        1194
Drug            1128
Public Order     987
Name: count, dtype: int64


### Duplicate Check — COMPAS after adaptation

In [14]:
exact_dups = compas.duplicated().sum()
feat_dups  = compas.duplicated(subset=BASE_COLS).sum()
print(f"COMPAS exact duplicates:        {exact_dups} ({exact_dups/len(compas)*100:.1f}%)")
print(f"COMPAS feature duplicates:      {feat_dups}  ({feat_dups/len(compas)*100:.1f}%)")

iowa_types   = set(iowa["offense_type"].unique())
compas_types = set(compas["offense_type"].unique())
print(f"\nIowa offense_type:   {sorted(iowa_types)}")
print(f"COMPAS offense_type: {sorted(compas_types)}")
assert iowa_types == compas_types or compas_types.issubset(iowa_types), "Mismatch!"
print("\n✓ offense_type categories match")
compas.head(3)

COMPAS exact duplicates:        4256 (59.0%)
COMPAS feature duplicates:      6597  (91.4%)

Iowa offense_type:   ['Drug', 'Other', 'Property', 'Public Order', 'Violent']
COMPAS offense_type: ['Drug', 'Other', 'Property', 'Public Order', 'Violent']

✓ offense_type categories match


,race,age_cat,offense_classification,offense_type,offense_subtype,priors_count,juv_fel_count,juv_misd_count,juv_other_count,recidivism,source
0,Other,Greater than 45,F,Violent,Assault,0,0,0,0,0,compas
1,Black,25 - 45,F,Violent,Assault,0,0,0,0,1,compas
2,Black,Less than 25,F,Drug,Drug Possession,4,0,0,1,1,compas


## 4. Sample Count Features for Iowa
Conditioned on `(offense_type, offense_classification, age_cat)` — **no race**, so no racial skew is injected.

In [15]:
EXTRA_COLS = ["priors_count", "juv_fel_count", "juv_misd_count", "juv_other_count"] #Columns to generate for Iowa

compas_ref = compas.dropna(subset=EXTRA_COLS).copy()
for col in EXTRA_COLS:
    compas_ref[col] = compas_ref[col].clip(lower=0).round().astype(int)

# ── Building lookup tables — conditioned on offense_type ───────────
lookups = {}
for col in EXTRA_COLS:
    G3 = compas_ref.groupby(["offense_type", "offense_classification", "age_cat"])[col].apply(list).to_dict()
    G2 = compas_ref.groupby(["offense_type", "offense_classification"])[col].apply(list).to_dict() #If age has NA
    G1 = compas_ref.groupby(["offense_classification"])[col].apply(list).to_dict() #If age and offense type has NA
    G0 = compas_ref[col].tolist()
    lookups[col] = (G3, G2, G1, G0)

def sample_count(row, col): #Select randomly from vector
    G3, G2, G1, G0 = lookups[col]
    k3 = (row["offense_type"], row["offense_classification"], row["age_cat"])
    k2 = (row["offense_type"], row["offense_classification"])
    k1 =  row["offense_classification"]
    pool = G3.get(k3) or G2.get(k2) or G1.get(k1) or G0
    return int(rng.choice(pool))

for col in EXTRA_COLS:
    iowa[col] = iowa.apply(lambda row: sample_count(row, col), axis=1)

print("Iowa count features generated.")
print(iowa[EXTRA_COLS].describe().round(2))

Iowa count features generated.
       priors_count  juv_fel_count  juv_misd_count  juv_other_count
count      25985.00       25985.00        25985.00         25985.00
mean           3.85           0.06            0.10             0.10
std            5.19           0.43            0.55             0.49
min            0.00           0.00            0.00             0.00
25%            0.00           0.00            0.00             0.00
50%            2.00           0.00            0.00             0.00
75%            5.00           0.00            0.00             0.00
max           38.00          20.00           13.00            17.00


### Duplicate Check — Iowa after count feature generation

In [17]:
BASE_COLS_FINAL = ["race", "age_cat", "offense_classification",
                   "offense_type", "offense_subtype", "recidivism"]
ALL_COLS        = BASE_COLS_FINAL + EXTRA_COLS

exact_dups = iowa.duplicated(subset=ALL_COLS).sum()
feat_dups  = iowa.duplicated(subset=BASE_COLS_FINAL).sum()
print(f"Iowa exact duplicates (all cols):     {exact_dups} ({exact_dups/len(iowa)*100:.1f}%)")
print(f"Iowa feature duplicates (no counts):  {feat_dups}  ({feat_dups/len(iowa)*100:.1f}%)")
iowa.head(3)

Iowa exact duplicates (all cols):     18918 (72.8%)
Iowa feature duplicates (no counts):  25143  (96.8%)


,race,age_cat,offense_classification,offense_type,offense_subtype,recidivism,source,priors_count,juv_fel_count,juv_misd_count,juv_other_count
0,White,Less than 25,F,Violent,Assault,1,iowa,3,0,0,0
1,White,Greater than 45,F,Public Order,OWI,1,iowa,4,0,0,0
2,White,25 - 45,F,Property,Burglary,1,iowa,2,0,0,0


## 5. Save — 6 Output Files

In [19]:
iowa[BASE_COLS_FINAL].to_csv(OUT + "iowa_adapted.csv", index=False)
iowa[BASE_COLS_FINAL + EXTRA_COLS].to_csv(OUT + "iowa_adapted_with_counts.csv", index=False)
compas[BASE_COLS_FINAL].to_csv(OUT + "compas_adapted.csv", index=False)
compas[BASE_COLS_FINAL + EXTRA_COLS].to_csv(OUT + "compas_adapted_with_counts.csv", index=False)

iowa_part   = iowa[BASE_COLS_FINAL + ["source"]].copy()
compas_part = compas[BASE_COLS_FINAL + ["source"]].copy()
combined    = pd.concat([iowa_part, compas_part], ignore_index=True)
combined    = combined.sample(frac=1, random_state=42).reset_index(drop=True)
combined.to_csv(OUT + "combined_adapted.csv", index=False)

iowa_part_c   = iowa[BASE_COLS_FINAL + EXTRA_COLS + ["source"]].copy()
compas_part_c = compas[BASE_COLS_FINAL + EXTRA_COLS + ["source"]].copy()
combined_counts = pd.concat([iowa_part_c, compas_part_c], ignore_index=True)
combined_counts = combined_counts.sample(frac=1, random_state=42).reset_index(drop=True)
combined_counts.to_csv(OUT + "combined_adapted_with_counts.csv", index=False)

print("Files saved!")

Files saved!


### Duplicate Check — Combined datasets

In [20]:
for name, df in [
    ("combined_adapted",             combined),
    ("combined_adapted_with_counts", combined_counts),
]:
    exact = df.duplicated().sum()
    feat  = df.duplicated(subset=BASE_COLS_FINAL).sum()
    print(f"{name}")
    print(f"  Exact duplicates:   {exact} ({exact/len(df)*100:.1f}%)")
    print(f"  Feature duplicates: {feat}  ({feat/len(df)*100:.1f}%)")
    print(f"  Source breakdown:   {df['source'].value_counts().to_dict()}\n")

combined_adapted
  Exact duplicates:   31740 (95.6%)
  Feature duplicates: 32124  (96.8%)
  Source breakdown:   {'iowa': 25985, 'compas': 7214}

combined_adapted_with_counts
  Exact duplicates:   23174 (69.8%)
  Feature duplicates: 32124  (96.8%)
  Source breakdown:   {'iowa': 25985, 'compas': 7214}



## 6. Summary

In [21]:
print("=" * 60)
print("FILES GENERATED")
print("=" * 60)
for name, df in [
    ("iowa_adapted.csv",                 iowa[BASE_COLS_FINAL]),
    ("iowa_adapted_with_counts.csv",     iowa[BASE_COLS_FINAL + EXTRA_COLS]),
    ("compas_adapted.csv",               compas[BASE_COLS_FINAL]),
    ("compas_adapted_with_counts.csv",   compas[BASE_COLS_FINAL + EXTRA_COLS]),
    ("combined_adapted.csv",             combined),
    ("combined_adapted_with_counts.csv", combined_counts),
]:
    print(f"  {name:45s} shape={df.shape}")

print("\nCOMPAS recidivism rate by race:")
print(compas.groupby("race")["recidivism"].mean().round(3).to_string())
print("\nCOMPAS avg priors by race:")
print(compas.groupby("race")["priors_count"].mean().round(2).to_string())
print("\nNaN check combined_with_counts:")
print(combined_counts.isnull().sum())

FILES GENERATED
  iowa_adapted.csv                              shape=(25985, 6)
  iowa_adapted_with_counts.csv                  shape=(25985, 10)
  compas_adapted.csv                            shape=(7214, 6)
  compas_adapted_with_counts.csv                shape=(7214, 10)
  combined_adapted.csv                          shape=(33199, 7)
  combined_adapted_with_counts.csv              shape=(33199, 11)

COMPAS recidivism rate by race:
race
Asian              0.281
Black              0.514
Hispanic           0.364
Native American    0.556
Other              0.353
White              0.394

COMPAS avg priors by race:
race
Asian              1.44
Black              4.44
Hispanic           2.25
Native American    6.00
Other              1.88
White              2.59

NaN check combined_with_counts:
race                      0
age_cat                   0
offense_classification    0
offense_type              0
offense_subtype           0
recidivism                0
priors_count              0